# RetinAI CNN v2 — auditoría de imágenes
Copia de trabajo creada a partir de Modelo_CheviFlow_V2.ipynb. La libreta original permanece intacta.

Ejecuta las celdas en orden. Esta fase inventaría archivos y genera muestras para revisión humana; **no entrena ni califica clínicamente imágenes automáticamente**. Comienza con ODIR y Sylhet. Después activa las otras cuatro fuentes para detectar variantes y duplicados. No compartas claves de Kaggle en el chat.

In [ ]:
%pip -q install kagglehub pillow pandas numpy matplotlib

In [ ]:
from pathlib import Path
import kagglehub

FUENTES = {
    "odir_original": "andrewmvd/ocular-disease-recognition-odir5k",
    "sylhet": "mahin661/eye-disease-classification-fundus-image-dataset",
    "cuatro_original": "gunavenkatdoddi/eye-diseases-classification",
    "cuatro_procesado": "gunavenkatdoddi/preprocessed-eye-diseases-fundus-images",
    "odir_carpetas": "tanjemahamed/odir5k-classification",
    "odir_clahe": "efiyearcan/odir-5k-preprocessed-with-clahe",
}
DESCARGAR_TODOS = False  # Cambia a True tras revisar espacio disponible en Colab.
seleccionadas = FUENTES if DESCARGAR_TODOS else {
    clave: FUENTES[clave] for clave in ("odir_original", "sylhet")
}
rutas = {}
for nombre, referencia in seleccionadas.items():
    print("Descargando:", nombre, referencia)
    rutas[nombre] = Path(kagglehub.dataset_download(referencia))
    print("Ruta:", rutas[nombre])
print("Fuentes disponibles:", list(rutas))

In [ ]:
from PIL import Image, ImageStat, UnidentifiedImageError
import hashlib
import pandas as pd
import re

EXTENSIONES = {".jpg", ".jpeg", ".png"}
registros = []
for fuente, carpeta in rutas.items():
    archivos = sorted(p for p in carpeta.rglob("*") if p.is_file() and p.suffix.lower() in EXTENSIONES)
    print(f"{fuente}: {len(archivos)} imágenes")
    for ruta in archivos:
        relativo = ruta.relative_to(carpeta)
        datos = {
            "fuente": fuente,
            "ruta": str(ruta),
            "relativo": str(relativo),
            "archivo": ruta.name,
            "etiqueta_carpeta": ruta.parent.name,
            "bytes": ruta.stat().st_size,
            "id_paciente_sugerido": ruta.stem.split("_")[0] if re.match(r"^\d+_(?:left|right)", ruta.stem, re.I) else None,
            "sha256": None, "ancho": None, "alto": None,
            "brillo_medio": None, "rojo_medio": None,
            "verde_medio": None, "azul_medio": None, "error": None,
        }
        try:
            digest = hashlib.sha256()
            with ruta.open("rb") as entrada:
                for bloque in iter(lambda: entrada.read(1024 * 1024), b""):
                    digest.update(bloque)
            datos["sha256"] = digest.hexdigest()
            with Image.open(ruta) as imagen:
                imagen.load()
                datos["ancho"], datos["alto"] = imagen.size
                muestra = imagen.convert("RGB")
                muestra.thumbnail((256, 256))
                estadisticas = ImageStat.Stat(muestra)
                datos["rojo_medio"], datos["verde_medio"], datos["azul_medio"] = [round(v, 2) for v in estadisticas.mean]
                datos["brillo_medio"] = round(ImageStat.Stat(muestra.convert("L")).mean[0], 2)
        except (OSError, ValueError, UnidentifiedImageError) as error:
            datos["error"] = str(error)[:200]
        registros.append(datos)
inventario = pd.DataFrame(registros)
display(inventario.groupby("fuente").agg(
    imagenes=("ruta", "size"),
    errores=("error", lambda s: s.notna().sum()),
    ancho_min=("ancho", "min"),
    alto_min=("alto", "min"),
    brillo_mediano=("brillo_medio", "median"),
))
inventario.to_csv("/content/retinai_inventario.csv", index=False)
print("Inventario guardado en /content/retinai_inventario.csv")

In [ ]:
# Las métricas de color y brillo son señales para inspección, no diagnósticos ni etiquetas de calidad.
duplicados = inventario[inventario.duplicated("sha256", keep=False) & inventario["sha256"].notna()]
print("Duplicados exactos:", len(duplicados))
display(duplicados.sort_values("sha256")[["fuente", "archivo", "etiqueta_carpeta", "sha256"]].head(50))
print("Distribución por carpeta:")
display(inventario.groupby(["fuente", "etiqueta_carpeta"]).size().rename("imagenes").reset_index())
print("Imágenes más oscuras (revisión manual):")
display(inventario.nsmallest(30, "brillo_medio")[["fuente", "archivo", "etiqueta_carpeta", "ancho", "alto", "brillo_medio"]])
print("Dimensiones menores que el mínimo actual de la interfaz (500 px por lado):")
display(inventario[(inventario["ancho"] < 500) | (inventario["alto"] < 500)].groupby("fuente").size())

In [ ]:
# Muestras aleatorias para detectar fotos externas del ojo y problemas de etiqueta.
import matplotlib.pyplot as plt

def mostrar_muestra(fuente, etiqueta=None, cantidad=24, semilla=42):
    filas = inventario[inventario["fuente"].eq(fuente) & inventario["error"].isna()]
    if etiqueta is not None:
        filas = filas[filas["etiqueta_carpeta"].str.lower().eq(etiqueta.lower())]
    filas = filas.sample(n=min(cantidad, len(filas)), random_state=semilla)
    if filas.empty:
        print("Sin imágenes para", fuente, etiqueta)
        return
    columnas = 6
    fig, ejes = plt.subplots((len(filas) + columnas - 1) // columnas, columnas, figsize=(15, 3 * ((len(filas) + columnas - 1) // columnas)))
    for eje in ejes.flat:
        eje.axis("off")
    for eje, (_, fila) in zip(ejes.flat, filas.iterrows()):
        with Image.open(fila["ruta"]) as imagen:
            eje.imshow(imagen.convert("RGB"))
        eje.set_title(f'{fila["etiqueta_carpeta"]}: {fila["archivo"][:22]}', fontsize=8)
    plt.tight_layout()
    plt.show()

mostrar_muestra("sylhet")
# Ejemplo: mostrar_muestra("sylhet", "cataract", 30)
# Tras activar DESCARGAR_TODOS: mostrar_muestra("odir_carpetas", "cataract")

## Siguiente decisión
Revisar manualmente las muestras, especialmente catarata de Sylhet, imágenes oscuras y tonos extremos. Los duplicados transformados requieren comparación visual o perceptual adicional; SHA-256 solo encuentra archivos idénticos. Para una evaluación válida, separar por paciente y origen antes de entrenar. Las cuatro etiquetas clínicas no sirven por sí solas para enseñar calidad ni para distinguir retinas de fotografías ajenas.

In [ ]:
from google.colab import files
files.download('/content/retinai_inventario.csv')

## Depuración automática y revisión humana
Ejecuta primero las celdas anteriores para generar `inventario`. Esta etapa **no asigna diagnósticos**. Aparta archivos idénticos con etiquetas incompatibles y las fotografías externas del ojo que pueden estar mezcladas en catarata. Los originales de ODIR requieren unir sus etiquetas oficiales antes de entrenar.

In [ ]:
import pandas as pd

# El hash identifica archivos idénticos; no detecta versiones recoloreadas o redimensionadas.
grupos = inventario.groupby("sha256", dropna=True)
etiquetas_por_hash = grupos["etiqueta_carpeta"].transform("nunique")
apariciones_por_hash = grupos["ruta"].transform("size")
revision = inventario.copy()
revision["estado_revision"] = "pendiente_verificacion"
revision["motivo_revision"] = ""
revision.loc[revision["error"].notna(), ["estado_revision", "motivo_revision"]] = ["excluir", "archivo ilegible"]
revision.loc[(revision["fuente"] == "odir_original") & (revision["etiqueta_carpeta"] == "preprocessed_images"),
             ["estado_revision", "motivo_revision"]] = ["variante_odir", "copia reducida del original; conservar vínculo al paciente"]
revision.loc[(revision["fuente"] == "odir_original") & (revision["etiqueta_carpeta"] == "Testing Images"),
             ["estado_revision", "motivo_revision"]] = ["reserva_prueba", "prueba original ODIR; no usar para entrenar"]
revision.loc[(revision["fuente"] == "odir_original") & (revision["etiqueta_carpeta"] == "Training Images"),
             ["estado_revision", "motivo_revision"]] = ["pendiente_etiqueta", "unir con etiquetas oficiales por paciente y ojo"]
revision.loc[(revision["fuente"] == "sylhet") & (revision["etiqueta_carpeta"] == "cataract"),
             ["estado_revision", "motivo_revision"]] = ["revision_visual", "carpeta mezcla fotos de retina y fotos externas del ojo"]
revision.loc[(revision["fuente"] == "sylhet") & (revision["etiqueta_carpeta"] != "cataract"),
             ["estado_revision", "motivo_revision"]] = ["candidato_provisional", "verificar anatomía, calidad y procedencia"]
conflicto = (revision["fuente"] == "sylhet") & (apariciones_por_hash > 1) & (etiquetas_por_hash > 1)
revision.loc[conflicto, ["estado_revision", "motivo_revision"]] = ["cuarentena", "archivo idéntico con etiquetas incompatibles"]
revision["decision_humana"] = ""  # completar tras observar la imagen, sin inventar etiquetas
revision["nota_humana"] = ""
display(revision.groupby(["fuente", "estado_revision"]).size().rename("imagenes").reset_index())
print("Conflictos por hash:", int(conflicto.sum()), "archivos en", revision.loc[conflicto, "sha256"].nunique(), "grupos")
print("Estos estados son para auditoría; candidato_provisional NO significa apto para entrenamiento.")
revision.to_csv("/content/retinai_revision.csv", index=False)
revision.loc[conflicto].to_csv("/content/retinai_conflictos.csv", index=False)
print("Archivos guardados: /content/retinai_revision.csv y /content/retinai_conflictos.csv")